# 2.3 Local Site

`LocalSite` runs the fast solver on the current machine through local workers. It is the best development path when `FS_SOLVER_PATH` points to an installed solver executable. This notebook runs the same complete job locally, inspects logs/results, plots traces, and lists generated output files.

By the end, you should be able to use `LocalSite` for small solver smoke tests, inspect local logs/results, and decide when a job needs AWS or HPC resources instead.


## How To Read This Tutorial

`LocalSite` is the tight feedback loop for authoring and smoke testing. It is not meant to make a laptop into a production cluster; it is meant to catch Python-side export mistakes, solver-path mistakes, and small-model physics issues before the same job is sent to AWS or HPC.

Read this notebook as the local version of the same site contract used remotely: submit a project-owned job, wait for a result, inspect logs and run metadata, and read traces through `TraceDataset`. The workflow is deliberately strict so failures remain visible.

## Configuration Checklist

A local site is useful for development and small checks only when the fast solver is installed on the machine running the notebook. The Python objects stay deliberately consistent across sites: a `Project` owns simulations and jobs, a `Job` describes work, and a `Site` decides where that work runs.

| Concern | AWS site | HPC site | Local site |
| --- | --- | --- | --- |
| Solver availability | Managed in the cloud environment. | Must be installed on the target cluster. | Must be installed on this workstation. |
| Staging | Uploads project/job artifacts to cloud storage. | Copies or syncs artifacts to a remote work directory. | Writes artifacts directly under the project path. |
| Execution | Submitted through the cloud service. | Submitted through the scheduler, usually SLURM. | Runs a local worker process. |
| Results | Fetched from cloud storage through the result API. | Fetched from the remote work directory. | Read directly from local result directories. |
| Best first use | Production and normal user workflows. | Institution-managed clusters with solver access. | API development, smoke tests, and small examples. |

The run cells below are intentionally strict. If `FS_SOLVER_PATH`, local worker startup, or solver execution is wrong, the cell should fail with enough site/job context to inspect logs rather than hiding the problem behind a broad exception handler.


## Site Mental Model

A site changes where a completed job runs; it should not change how the simulation is authored or how results are read. Keep the project, simulation, acquisition, jobs, trace reads, and ParaView output requests ordinary. Let the site object handle authentication, staging, scheduling, storage, polling, and fetching.

That separation is what makes it possible to prototype locally, run production jobs in the cloud or on HPC, and keep the analysis cells nearly identical.


## Run Handles, Futures, And Task Summaries

`LocalSite` uses local workers behind the same run-handle interface used by remote sites. The handle lets the notebook submit work, wait for completion, fetch logs, and read traces without depending on Dask internals.

The completion line includes task counts. This is especially useful for time-domain jobs because they expand to many frequency-domain tasks; one failed frequency should be visible in the summary even if the overall run reached a terminal state.

## Imports And Shared Job Builder

The helper creates one simulation and two jobs: a time-domain trace job and a single-frequency ParaView QC job. That pair is repeated across all site tutorials so the only moving part is the `Site` object.

The builder returns project-owned jobs rather than loose JSON. Calling `site.submit(job)` serializes the simulation, job, acquisition, mesh, outputs, and units into the project structure before staging or execution. That is the core site contract users should remember: author locally, submit through the selected site, fetch through the returned result handle.

The QC job uses `upscale=0` for a native mesh view and faster local iteration. Increase it only when the visualization, not execution plumbing, is the focus.


In [ ]:
import os

import numpy as np
import frequensolve as fs

u = fs.ureg


In [ ]:
def build_acoustic_tutorial_jobs(project_path, *, simulation_name, trace_job_name, qc_job_name, f_max=25.0):
    project = fs.Project(
        name="project",
        pretty_name=simulation_name,
        path=project_path,
        log_level="INFO",
        log_to_console=True,
    )
    sim = project.new_simulation(
        name=simulation_name,
        physics="acoustic",
        dimension=2,
        units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
    )

    model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
    model.add_surface(name="top", depth=0.0 * u.km)
    model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
    model.add_surface(name="interface", depth=0.22 * u.km)
    model.add_layer(name="basement", properties={"Vp": 2.4 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3})
    model.add_surface(name="bottom", depth=0.5 * u.km)
    sim += model

    sim += model.hex_mesh_generator([8, 4])
    sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=f_max)
    sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
    sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

    acq = fs.Acquisition()
    acq.add_sources(kind="scalar", coords=[[0.35, 0.05], [0.65, 0.05]])
    hydrophone = fs.ReceiverNode(name="hydrophone")
    hydrophone.add_component(name="p", field="pressure")
    acq.add_receiver_group(name="surface", device=hydrophone, coords=[[x, 0.04] for x in np.linspace(0.1, 0.9, 61)])
    sim += acq
    sim += fs.Discretization()
    sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

    trace_job = fs.TimeDomainJob(
        name=trace_job_name,
        simulation=sim,
        f_min=0.0,
        f_max=f_max,
        T_max=0.9,
    )
    qc_job = fs.FrequencyDomainJob(
        name=qc_job_name,
        simulation=sim,
        f_list=[12.0],
        outputs=[
            fs.ParaviewOutput(
                name="pv_qc",
                fields=["pressure"],
                properties=["vp", "rho", "Subdomain"],
                show_pml=True,
                upscale=0,
                order=1,
            )
        ],
    )
    return project, sim, trace_job, qc_job


## Check Local Solver Configuration

The Python package can author projects without a solver, but local execution requires `FS_SOLVER_PATH`. The run cell remains strict so a missing or invalid solver path fails clearly.


In [ ]:
solver_path = os.environ.get("FS_SOLVER_PATH")
solver_path


## Run The Local Jobs

The local trace and QC jobs use separate `LocalSite` instances with `shutdown_on_completion=True`. That keeps the example explicit and avoids reusing a worker after it has intentionally been shut down.


In [ ]:
project, sim, trace_job, qc_job = build_acoustic_tutorial_jobs(
    "./scratch/tutorials/local_site",
    simulation_name="local_site_acoustic",
    trace_job_name="time_local_site",
    qc_job_name="freq_local_site_qc",
)

trace_site = fs.LocalSite(
    n_workers=2,
    threads_per_worker=1,
    shutdown_on_completion=True,
    verbose=True,
)
trace_result = trace_site.submit(trace_job).wait()

qc_site = fs.LocalSite(
    n_workers=2,
    threads_per_worker=1,
    shutdown_on_completion=True,
    verbose=True,
)
qc_result = qc_site.submit(qc_job).wait()
{
    "trace_status": trace_result.status,
    "qc_status": qc_result.status,
}


## Inspect Local Logs, Run Metadata, And Outputs

The local result directory contains the same artifacts that remote sites fetch back: logs, trace HDF5 files, run state, and requested ParaView files.


In [ ]:
logs = trace_result.logs()
qc_outputs = qc_result.output_files(existing=True)
traces = trace_result.traces(upscale=4)
{
    "trace_successful": trace_result.successful,
    "qc_successful": qc_result.successful,
    "logs": str(logs),
    "qc_output_count": len(qc_outputs),
    "trace_files": traces.files,
    "frequency_summary": trace_job.frequency_summary(),
}


## Plot Local Trace Results

The local result is read through the same `TraceDataset` facade used by remote sites. This is the quick check that the exported acquisition, solver run, trace storage, and Python reader agree before moving to larger infrastructure.

In [ ]:
traces = trace_result.traces(upscale=4)
wavelet = fs.RickerWavelet(f=12.0)
group = traces.groups[0]
component = traces.components(group)[0]
source = traces.sources(group)[0]
gather = traces.td(group, component, source, wavelet, upscale=4, T_max=0.9)
fs.plot_gather(
    gather,
    A=2.0 * np.nanstd(np.real(gather.values)),
    cmap="gray",
    figsize=(9, 4),
    title=f"{trace_job.name}: {group}/{component}/source {source}",
)


## Before Moving On

Local runs are most useful when the model is small, the solver build is known, and the goal is rapid iteration. The result summary, logs, and task counts should be checked every time. A completed local run can still contain failed frequency tasks, so use the printed succeeded/failed counts and the run manifest before trusting a plot.

When the local workflow is clean but the model is too large, move the same saved job to AWS or HPC rather than rewriting the simulation.

## Result Review Checklist

A local site is the fastest place to catch Python-side and export-side mistakes, but it is only representative of production if the same solver build and runtime options are available locally.

| Artifact | What to confirm |
| --- | --- |
| `FS_SOLVER_PATH` | Points to the intended solver executable before the strict run cell starts. |
| Worker settings | `n_workers` and `threads_per_worker` fit the local machine and the test job size. |
| Project directory | Logs, job JSON, trace files, and ParaView files are written under the tutorial scratch path. |
| Trace plot | The local result can be read through `TraceDataset`, proving the output path is complete. |

Use local runs for small authoring checks and visualization smoke tests. Use AWS or HPC sites for jobs where memory, solver installation, or runtime makes local execution unrealistic.
